In this notebook I  put together all the possible pools for which I have or should collect place json.
First we collect pools from google maps, then I cross check with pools from baederleben website. if the pool is not in out list I search google maps manually for it.

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

#### Baederleben pools

In [ ]:
baederleben_pools = pd.read_csv("//home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/baederleben_pools_07042026.csv")
baederleben_pools.info()

In [ ]:
# merge duplicates so we have the alternatives for the columns with different values
baederleben_pools = baederleben_pools.groupby(["longitude", "latitude", "postalcode", "city"], as_index=False).agg({
    "pool_id": "first",
    "pool_type": lambda x: ", ".join(x.dropna().unique()),
    "pool_name": lambda x: ", ".join(x.dropna().unique()),
    "municipal_code": "first",
    "street": lambda x: ", ".join(x.dropna().unique()),
    "pool_website": lambda x: ", ".join(x.dropna().unique())
})
baederleben_pools.info()

### All possible pools to collect place jsons on

In [ ]:
collected_pools = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/all_inclusive_possible_pools_to_use_in_project_09042026.csv")
collected_pools.info()

In [ ]:
manually_collected_pools = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/all_manual_search_baederleben_pools_24042026.csv")
manually_collected_pools.info()

In [ ]:
all_matched_pools = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/all_matched_baederleben_pools_address_radius.csv")
all_matched_pools.info()

### How many of the baederleben are in our list of pools to collect data on

In [ ]:
to_collected_pools = pd.concat([collected_pools, manually_collected_pools, all_matched_pools], ignore_index=True)[['pool_id', 'pool_name', 'address_admin', 'place_id', 'unique_identifier']]

to_collected_pools.info()

In [ ]:
missing_pairs = (
    baederleben_pools[['pool_id', 'pool_name']]
    .merge(
        to_collected_pools[['pool_id', 'pool_name']].drop_duplicates(),
        on=['pool_id'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns='_merge')
)
# missing_pairs['address'] = missing_pairs['street'] + ", " + missing_pairs['postalcode'].astype(str) + " " + missing_pairs['city']
missing_pairs#.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/still_missing_baederleben_pools_24042026.csv", index=False)

In [ ]:
# Diagnose pool_id vs pool_name mismatches
import re

def normalize_pool_name(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_pool_id(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    # Harmonize values like "8787" and "8787.0"
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

bl = baederleben_pools.copy()
dc = to_collected_pools.copy()

bl["pool_id_norm"] = bl["pool_id"].map(normalize_pool_id)
dc["pool_id_norm"] = dc["pool_id"].map(normalize_pool_id)

bl["pool_name_norm"] = bl["pool_name"].map(normalize_pool_name)
dc["pool_name_norm"] = dc["pool_name"].map(normalize_pool_name)

print("Unique pool_id in baederleben:", bl["pool_id_norm"].nunique(dropna=True))
print("Unique pool_id in deduplicated:", dc["pool_id_norm"].nunique(dropna=True))
print("Unique pool_name in deduplicated:", dc["pool_name_norm"].nunique(dropna=True))

missing_by_id = bl.loc[~bl["pool_id_norm"].isin(dc["pool_id_norm"]), ["pool_id_norm", "pool_name"]]
missing_by_name = bl.loc[~bl["pool_name_norm"].isin(dc["pool_name_norm"]), ["pool_id_norm", "pool_name", "pool_name_norm"]]

print("Missing by pool_id:", len(missing_by_id))
print("Missing by pool_name:", len(missing_by_name))

# When IDs match but names do not, names are different labels for the same ID.
name_gap_with_id_present = missing_by_name[missing_by_name["pool_id_norm"].isin(dc["pool_id_norm"])]
print("Missing-by-name rows whose pool_id is present:", len(name_gap_with_id_present))

name_to_id_counts = (
    dc.dropna(subset=["pool_name_norm", "pool_id_norm"])
    .groupby("pool_name_norm")["pool_id_norm"]
    .nunique()
    .reset_index(name="id_count")
)

names_used_by_multiple_ids = name_to_id_counts[name_to_id_counts["id_count"] > 1]
ids_with_missing_name = dc[(dc["pool_id_norm"].notna()) & (dc["pool_name_norm"].isna())]["pool_id_norm"].nunique()

print("Normalized pool names used by multiple pool_id values:", len(names_used_by_multiple_ids))
print("pool_id values with missing pool_name:", ids_with_missing_name)

display(name_gap_with_id_present.head(40))
display(names_used_by_multiple_ids.head(20))

### All the baederleben pools have been matched to pre-existing google collected places or were manually searched. Now we create the main list of all possible pools.

### Deduplication

In [ ]:
uid_with_value = to_collected_pools[to_collected_pools['unique_identifier'].notna()].drop_duplicates(subset=['unique_identifier'], keep='first')
uid_missing = to_collected_pools[to_collected_pools['unique_identifier'].isna()].copy()

to_collect = pd.concat([uid_with_value, uid_missing], ignore_index=True)
to_collect.info()

In [ ]:
to_collect.shape

In [ ]:
to_collect[to_collect['place_id'].notnull()].drop_duplicates(subset=['place_id'], keep='first').shape

In [ ]:
to_collect.to_csv("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/pools_to_collect_24042026.csv", index=False)